# Semana 6 · ¿Cómo se midió el tamaño de la Tierra sin salir de ella?

Este cuaderno acompaña a la sesión de proyecto de la semana 6: reproducir el método de Eratóstenes con datos históricos y con medidas de una vara, y estudiar cuánto se equivoca el resultado cuando el ángulo se mide mal.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. De la sombra al ángulo

Una vara vertical de altura $h$ proyecta al mediodía solar una sombra de longitud $s$. Los rayos, la vara y la sombra forman un triángulo rectángulo, y el ángulo $z$ entre los rayos y la vertical (la **distancia cenital** del Sol) cumple

$$\tan z = \frac{s}{h} \quad\Rightarrow\quad z = \arctan\frac{s}{h}.$$

La **tangente** de un ángulo es el cociente entre el cateto opuesto y el adyacente, y la **arcotangente** es su operación inversa: dado el cociente, devuelve el ángulo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def distancia_cenital(h, s):
    """Ángulo (grados) entre los rayos del Sol y la vertical, para una vara de altura h y sombra s (misma unidad)."""
    return np.degrees(np.arctan(s / h))

print(distancia_cenital(1.00, 0.125))     # vara de 1 m con 12,5 cm de sombra
print(distancia_cenital(1.00, 1.00))      # sombra igual a la altura

**Ejercicio.** Una vara de 1,00 m proyecta 0,60 m de sombra. ¿Qué ángulo forman los rayos con la vertical?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
print(distancia_cenital(1.00, 0.60))

## 2. El método de Eratóstenes

Si los rayos del Sol son paralelos y dos lugares están en el mismo meridiano, el ángulo entre sus verticales es la diferencia entre sus distancias cenitales, $\Delta z$. Ese ángulo es la fracción $\Delta z/360^\circ$ de la circunferencia de la Tierra, y la distancia $d$ entre los lugares es esa misma fracción de la circunferencia $C$:

$$\frac{d}{C} = \frac{\Delta z}{360^\circ} \quad\Rightarrow\quad C = d\cdot\frac{360^\circ}{\Delta z}, \qquad R = \frac{C}{2\pi}.$$

In [ ]:
def circunferencia(delta_grados, distancia):
    """Circunferencia de la Tierra a partir de la diferencia angular y la distancia entre los dos lugares."""
    return distancia * 360 / delta_grados

# Datos tradicionales (Siena-Alejandría): 7,2° y 5000 estadios
C_estadios = circunferencia(7.2, 5000)
print(f"Circunferencia: {C_estadios:,.0f} estadios")

# La longitud del estadio es incierta: dos valores propuestos
for nombre, metros in [("estadio de unos 160 m", 160), ("estadio de unos 185 m", 185)]:
    C_km = C_estadios * metros / 1000
    print(f"{nombre}: {C_km:,.0f} km   (diferencia con 40 075 km: {100 * (C_km / 40075 - 1):+.1f} %)")

El resultado depende del valor del estadio, que los historiadores no conocen con certeza. Con 160 m el método da una circunferencia casi exacta; con 185 m, un 15 % mayor. El método es correcto, y el principal error está en las dos magnitudes: el ángulo y la distancia.

## 3. Qué pasa si el ángulo se mide mal

El ángulo $\Delta z$ es pequeño (unos grados) y la circunferencia es inversamente proporcional a él, así que un error pequeño en el ángulo produce un error grande en el resultado.

In [ ]:
d = 800      # km, distancia entre los dos lugares (valor de partida)
delta_real = d * 360 / 40008     # ángulo que correspondería con la circunferencia del meridiano terrestre (40 008 km)
print(f"Ángulo teórico para d = {d} km: {delta_real:.2f}°")

errores = np.linspace(-1, 1, 101)          # error de medida del ángulo, en grados
C = circunferencia(delta_real + errores, d)
plt.figure(figsize=(7, 3.5))
plt.plot(errores, C)
plt.axhline(40008, color="gray", ls="--", lw=1)
plt.xlabel("error al medir el ángulo (°)")
plt.ylabel("circunferencia obtenida (km)")
plt.title("Un error de 1° en un ángulo de 7° cambia el resultado entre un 12 % y un 16 %")
plt.grid(alpha=0.3)
plt.show()

Mide los ángulos con cuidado: esa es la parte de la sesión de astronomía.

## 4. Un ejemplo con datos de dos varas (inventados)

Dos grupos en el mismo meridiano, a 555 km de distancia, miden el mismo día en que el Sol está sobre el ecuador celeste (equinoccio), cada uno al mediodía solar, con una vara de 1,000 m:

- Grupo A: sombra de 0,842 m.
- Grupo B: sombra de 0,995 m.

In [ ]:
zA = distancia_cenital(1.000, 0.842)
zB = distancia_cenital(1.000, 0.995)
dz = zB - zA
C_ej = circunferencia(dz, 555)
print(f"zA = {zA:.2f}°, zB = {zB:.2f}°, diferencia = {dz:.2f}°")
print(f"Circunferencia estimada: {C_ej:,.0f} km  ({100 * (C_ej / 40008 - 1):+.1f} % respecto a 40 008 km)")

El ángulo real entre los dos lugares es de 5° (555 km de un meridiano de 40 008 km) y las medidas dan 4,76°: un error de unos 0,24° produce un error del 5 % en la circunferencia. Con una vara de 1 m, 0,24° son unos 7 mm de sombra: la precisión de tus medidas importa mucho.

## 5. Tus medidas: ¿cuál es tu latitud?

El mismo método da tu latitud. Al mediodía solar, la distancia cenital es $z = |\varphi - \delta|$, con $\delta$ la declinación del Sol ese día (la aproximación es la de la semana 1). Si estás en el hemisferio norte y la sombra apunta al norte, $\varphi = \delta + z$.

In [ ]:
OBLICUIDAD = 23.44
def declinacion(dia):
    return -OBLICUIDAD * np.cos(np.radians(360 / 365 * (dia + 10)))

# Tus medidas: cámbialas
altura_vara = 1.000      # m
sombra = 0.842           # m, medida al mediodía solar (la más corta del día)
dia_del_ano = 80         # 21 de marzo = 80; 21 de junio = 172; 23 de septiembre = 266; 21 de diciembre = 355

z = distancia_cenital(altura_vara, sombra)
print(f"Distancia cenital: {z:.2f}°")
print(f"Declinación del Sol ese día: {declinacion(dia_del_ano):.2f}°")
print(f"Latitud estimada (sombra hacia el norte): {declinacion(dia_del_ano) + z:.2f}°")

Dos avisos. La declinación de esta aproximación tiene un error de hasta 1°, y ese error pasa directamente a la latitud: para medidas más finas usa la declinación de una efeméride (Stellarium la muestra). Y el cálculo supone que la sombra apunta al norte; si apunta al sur, la latitud es $\delta - z$.

## 6. Para pensar

1. Si Alejandría y Siena no están en el mismo meridiano (hay unos 3° de diferencia de longitud), ¿por qué el método sigue funcionando aproximadamente?
2. Los rayos del Sol no son exactamente paralelos: el Sol tiene un diámetro aparente de 0,53°. ¿Cuánto error de ángulo puede introducir esto en cada medida?
3. Calcula cuánto cambia la circunferencia si la distancia medida con pasos tiene un error del 5 %.